In [1]:
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error
from tensorflow.python.keras.layers import Dropout
from torch.ao.nn.quantized.modules import dropout

In [2]:
url ="https://raw.githubusercontent.com/amankharwal/Website-data/master/CarPrice.csv"
df =  pd.read_csv(url)
print(df.head())


   car_ID  symboling                   CarName fueltype aspiration doornumber  \
0       1          3        alfa-romero giulia      gas        std        two   
1       2          3       alfa-romero stelvio      gas        std        two   
2       3          1  alfa-romero Quadrifoglio      gas        std        two   
3       4          2               audi 100 ls      gas        std       four   
4       5          2                audi 100ls      gas        std       four   

       carbody drivewheel enginelocation  wheelbase  ...  enginesize  \
0  convertible        rwd          front       88.6  ...         130   
1  convertible        rwd          front       88.6  ...         130   
2    hatchback        rwd          front       94.5  ...         152   
3        sedan        fwd          front       99.8  ...         109   
4        sedan        4wd          front       99.4  ...         136   

   fuelsystem  boreratio  stroke compressionratio horsepower  peakrpm citympg  \

In [3]:
df.shape

(205, 26)

In [4]:
df


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    str    
 3   fueltype          205 non-null    str    
 4   aspiration        205 non-null    str    
 5   doornumber        205 non-null    str    
 6   carbody           205 non-null    str    
 7   drivewheel        205 non-null    str    
 8   enginelocation    205 non-null    str    
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    str    
 15  cylindernumber    205 non-null    str    
 16  enginesize        205 non-null    int64  
 17  fuelsyst

In [6]:
df.isnull().sum().to_frame(name='nullcount')


,nullcount
car_ID,0
symboling,0
CarName,0
fueltype,0
aspiration,0
doornumber,0
carbody,0
drivewheel,0
enginelocation,0
wheelbase,0


In [9]:
df['company'] = df['CarName'].apply(lambda x: x.split(" ")[0])
df['company'].value_counts()
df['company'].unique()

<ArrowStringArray>
['alfa-romero',        'audi',         'bmw',   'chevrolet',       'dodge',
       'honda',       'isuzu',      'jaguar',       'maxda',       'mazda',
       'buick',     'mercury',  'mitsubishi',      'Nissan',      'nissan',
     'peugeot',    'plymouth',     'porsche',    'porcshce',     'renault',
        'saab',      'subaru',      'toyota',     'toyouta',   'vokswagen',
  'volkswagen',          'vw',       'volvo']
Length: 28, dtype: str

In [10]:
df.drop(columns=['CarName','car_ID'], inplace=True)

In [13]:
corrections = {
    'maxda': 'mazda',
    'porcshce': 'porsche',
    'toyouta': 'toyota',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen',
    'Nissan': 'nissan'
}

df['company'] = df['company'].replace(corrections)
df['company'].unique()

<ArrowStringArray>
['alfa-romero',        'audi',         'bmw',   'chevrolet',       'dodge',
       'honda',       'isuzu',      'jaguar',       'mazda',       'buick',
     'mercury',  'mitsubishi',      'nissan',     'peugeot',    'plymouth',
     'porsche',     'renault',        'saab',      'subaru',      'toyota',
  'volkswagen',  'volkswagon',       'volvo']
Length: 23, dtype: str

In [19]:
number = {
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'eight': 8,
    'twelve': 12
}

df['doornumber'] = df['doornumber'].map(number)
df['cylindernumber'] = df['cylindernumber'].map(number)

In [31]:
df

,symboling,doornumber,carbody,drivewheel,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,...,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,company,fueltype_gas,aspiration_std
0,3,2,convertible,rwd,88.6,168.8,64.1,48.8,2548,dohc,...,2.68,9.0,111,5000,21,27,13495.0,alfa-romero,1,1
1,3,2,convertible,rwd,88.6,168.8,64.1,48.8,2548,dohc,...,2.68,9.0,111,5000,21,27,16500.0,alfa-romero,1,1
2,1,2,hatchback,rwd,94.5,171.2,65.5,52.4,2823,ohcv,...,3.47,9.0,154,5000,19,26,16500.0,alfa-romero,1,1
3,2,4,sedan,fwd,99.8,176.6,66.2,54.3,2337,ohc,...,3.40,10.0,102,5500,24,30,13950.0,audi,1,1
4,2,4,sedan,4wd,99.4,176.6,66.4,54.3,2824,ohc,...,3.40,8.0,115,5500,18,22,17450.0,audi,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,4,sedan,rwd,109.1,188.8,68.9,55.5,2952,ohc,...,3.15,9.5,114,5400,23,28,16845.0,volvo,1,1
201,-1,4,sedan,rwd,109.1,188.8,68.8,55.5,3049,ohc,...,3.15,8.7,160,5300,19,25,19045.0,volvo,1,0
202,-1,4,sedan,rwd,109.1,188.8,68.9,55.5,3012,ohcv,...,2.87,8.8,134,5500,18,23,21485.0,volvo,1,1
203,-1,4,sedan,rwd,109.1,188.8,68.9,55.5,3217,ohc,...,3.40,23.0,106,4800,26,27,22470.0,volvo,0,0


In [28]:
df['fueltype_gas'] = df['fueltype'].apply(lambda x: 1 if x == 'gas' else 0)
df['aspiration_std'] = df['aspiration'].apply(lambda x: 1 if x == 'std' else 0)
df['enginelocation_front'] = df['enginelocation'].apply(lambda x: 1 if x == 'front' else 0)
df.drop(columns=['enginelocation','aspiration','fueltype'], inplace=True)

KeyError: 'fueltype'

In [30]:
df.drop(columns=['enginelocation_front'], inplace=True)

In [32]:
df['dw_fwd'] = df['drivewheel'].apply(lambda x: 1 if x == 'fwd' else 0)
df['dw_rwd'] = df['drivewheel'].apply(lambda x: 1 if x == 'rwd' else 0)
df['dw_4wd'] = df['drivewheel'].apply(lambda x: 1 if x == '4wd' else 0)
df.drop(columns=['drivewheel'], inplace=True)

In [36]:
df['carbody'].value_counts()


carbody
sedan          96
hatchback      70
wagon          25
hardtop         8
convertible     6
Name: count, dtype: int64

In [37]:
df = pd.get_dummies(df,drop_first=True)


In [38]:
df

,symboling,doornumber,wheelbase,carlength,carwidth,carheight,curbweight,cylindernumber,enginesize,boreratio,...,company_peugeot,company_plymouth,company_porsche,company_renault,company_saab,company_subaru,company_toyota,company_volkswagen,company_volkswagon,company_volvo
0,3,2,88.6,168.8,64.1,48.8,2548,4,130,3.47,...,False,False,False,False,False,False,False,False,False,False
1,3,2,88.6,168.8,64.1,48.8,2548,4,130,3.47,...,False,False,False,False,False,False,False,False,False,False
2,1,2,94.5,171.2,65.5,52.4,2823,6,152,2.68,...,False,False,False,False,False,False,False,False,False,False
3,2,4,99.8,176.6,66.2,54.3,2337,4,109,3.19,...,False,False,False,False,False,False,False,False,False,False
4,2,4,99.4,176.6,66.4,54.3,2824,5,136,3.19,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,4,109.1,188.8,68.9,55.5,2952,4,141,3.78,...,False,False,False,False,False,False,False,False,False,True
201,-1,4,109.1,188.8,68.8,55.5,3049,4,141,3.78,...,False,False,False,False,False,False,False,False,False,True
202,-1,4,109.1,188.8,68.9,55.5,3012,6,173,3.58,...,False,False,False,False,False,False,False,False,False,True
203,-1,4,109.1,188.8,68.9,55.5,3217,6,145,3.01,...,False,False,False,False,False,False,False,False,False,True


In [39]:
df.shape

(205, 61)

In [40]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

y = df["price"]
df.drop(columns=['price'], inplace=True)

x_train, x_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("x train shape ;",x_train_scaled.shape)
print("x test shape :",x_test_scaled.shape)



x train shape ; (164, 60)
x test shape : (41, 60)


In [42]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score , mean_absolute_error
regressor = LinearRegression()
regressor.fit(x_train_scaled, y_train)
y_pred = regressor.predict(x_test_scaled)


train_r2 = regressor.score(x_test_scaled, y_test)
test_r2 =  r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Train R² Score: {train_r2:.4f}")
print(f"Test R² Score:  {test_r2:.4f}")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")

Train R² Score: 0.8946
Test R² Score:  0.8946
Mean Absolute Error (MAE): $1940.99


In [49]:
import pandas as pd

# Model ke dimaag ki priority list nikaal rahe hain
priority_table = pd.DataFrame({
    'features': df.columns,
    'Importance_Score': regressor.coef_
})

# Sabse zyada importance wale upar rakho
print(priority_table.sort_values(by='Importance_Score', ascending=False).head(5))

           features  Importance_Score
8        enginesize       4358.636512
6        curbweight       2352.233061
28  enginetype_ohcf       2299.384095
31  fuelsystem_2bbl       1365.933220
2         wheelbase       1274.588503


In [51]:
actual  = y_test.iloc[0]
predict = regressor.predict(x_test_scaled[0].reshape(1, -1))[0]
error  = abs( actual - predict )

print(error)

1094.6555147354666


In [59]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout

# 1. Neural Network Architecture design karo
model_nn = Sequential([
    # Input Layer + Pehli Hidden Layer
    Dense(64, activation='relu', input_shape=(x_train_scaled.shape[1],)),
    Dropout(0.2),
    # Doosri Hidden Layer (Non-linear patterns seekhne ke liye)
    Dense(32, activation='relu'),

    # Output Layer (Sirf 1 number chahiye: Price)
    Dense(1)
])

# 2. Compile model (Loss function aur Optimizer)
model_nn.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 3. Model training (Epochs = Kitni baar data poora padhna hai)
history = model_nn.fit(
    x_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0 # Clean output ke liye
)

print("Training Complete!")

C:\Users\avupa\OneDrive\Desktop\all files\learning project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Complete!


In [60]:
from sklearn.metrics import r2_score

# Deep learning model se prediction lo
y_pred_nn = model_nn.predict(x_test_scaled).flatten()

test_r2_nn = r2_score(y_test, y_pred_nn)
print(f"Deep Learning Model R² Score: {test_r2_nn:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Deep Learning Model R² Score: 0.7595


In [61]:
import joblib

joblib.dump(regressor, 'model.pkl')

joblib.dump(scaler, 'scaler.pkl')

joblib.dump(df.columns.to_list(), 'columns.pkl')

print ("saved !!")

saved !!
